# 🔋 Energy & Power of an MX Systolic-Array Accelerator — a guided walkthrough

This notebook reproduces, **stage by stage**, exactly what
`rundir-accelergy/run_example_22nm.sh` does: estimate the **energy and power** of a custom
**MX (microscaling) systolic-array** AI accelerator running a small transformer block, at a
**22 nm** technology node.

It is written for people new to these tools. Three programs cooperate — each does *one* job:

| tool | its one job | analogy |
|---|---|---|
| **SCALE-Sim** | cycle-accurate **activity** simulator of a systolic array. Counts MAC-cycles and every SRAM/DRAM access. **Computes no energy.** | a stopwatch + tally counter |
| **Accelergy** | the **energy accountant**. Multiplies each activity count by an energy-per-action, and sums. | the spreadsheet |
| **CACTI** | **memory physics**. Given a memory's size / width / node, returns the real pJ per access. | the lab bench for memories |

**Division of labor for this design:**

```
   how many accesses / MAC-cycles   ←  SCALE-Sim     (activity, no energy)
   SRAM & DRAM energy per access     ←  CACTI         (real silicon @ 22 nm)
   the MAC / array energy per op     ←  your MX netlist (energy/pe_netlist.json)  ← injected at the end
```

Why the netlist? CACTI only models *memories*. The systolic array's multipliers are logic, which
CACTI can't estimate — so Accelergy fills them with a dummy `1.0 pJ`. The last stage **replaces that
dummy** with the real per-op energy from your synthesized MX array. That is the whole point of the flow.

## The 5-stage pipeline

Each stage below is one box. The arrows are files handed from one tool to the next.

```
 configs/scale_accel.cfg
          │
   [1] preprocess.py ─────────────►  architecture.yaml      (the hardware tree, stamped 22nm)
          │
   [2] scale.py ──────────────────►  *_REPORT.csv           (cycles + memory + per-PE RF access)
          │
   [3] create_action_count.py ────►  action_count.yaml      (counts per component)
          │
   [4] accelergy (+ CACTI) ───────►  ERT.yaml,              (pJ per action)
          │                          energy_estimation.yaml (pJ per component)
          │
   [5] accelergy_eval.py ─────────►  ACCELERGY_SUMMARY.md   (swap dummy MAC → MX netlist → Watts)
```

Run the cells in order. The whole thing takes ~1–2 minutes (stage 2 is the slow one).

In [ ]:
# === Setup: paths, knobs, and a small shell helper =========================
import os, subprocess, json, pathlib, shutil
import yaml
import pandas as pd

# --- EDIT these two if your checkout / venv live elsewhere -------------------
ROOT = os.environ.get("SCALESIM_ROOT", "/home/xinting/Desktop/scale-sim-v3-mx")
# Toolchain env (python3 + accelergy + cacti all live in <VENV>/bin). Defaults to the
# conda env built for this experiment; override with SCALESIM_VENV=... if yours differs.
VENV = os.environ.get("SCALESIM_VENV", "/home/xinting/miniconda3/envs/scalesim-mx")
HERE = os.path.join(ROOT, "rundir-accelergy")
# Activity engine is THIS repo's own scale.py (scale-sim-v3): it emits the per-PE register-file
# access counts (cols 19-24) and REPEAT_CYCLE, so pe_regfile is measured, not fabricated.

# knobs — identical to the shell script's defaults
TECH     = "22nm"
CLOCK_HZ = 1e9
CFG  = os.path.join(ROOT, "configs", "scale_accel.cfg")
TOPO = os.path.join(ROOT, "topologies", "GEMM_mnk", "transformer_partial.csv")
KIND = "gemm"

# run_name is read from the config (the shell script greps the same line)
RUN_NAME = next(l.split("=", 1)[1].strip() for l in open(CFG)
                if l.strip().lower().startswith("run_name"))

SCSIM  = f"/tmp/{RUN_NAME}_scsim"            # where SCALE-Sim writes raw reports
OUT    = f"/tmp/{RUN_NAME}_out"             # where finished artifacts are collected
SC_RAW = f"{SCSIM}/{RUN_NAME}"             # reports BEFORE stage 3 moves them
SC_OUT = f"{OUT}/scale_sim_output_{RUN_NAME}"   # reports AFTER stage 3
AC_OUT = f"{OUT}/accelergy_output_{RUN_NAME}"   # Accelergy/CACTI outputs

# make DataFrames print nicely even outside Jupyter (e.g. when validating as a script)
try:
    display
except NameError:
    def display(x):
        print(x.to_string(index=False) if isinstance(x, pd.DataFrame) else x)

def sh(cmd, cwd=HERE, show="full", tail=30):
    """Run `cmd` with the venv on PATH (so python3/accelergy/cacti resolve there) and the
    fork on PYTHONPATH. show: 'full' | 'tail' | None | a predicate(line)->bool to filter."""
    env = os.environ.copy()
    env["PATH"] = VENV + "/bin:" + env["PATH"]
    env["PYTHONPATH"] = ROOT + ((":" + env["PYTHONPATH"]) if env.get("PYTHONPATH") else "")
    p = subprocess.run(["bash", "-c", cmd], cwd=cwd, env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = p.stdout
    if show == "full":   print(out.rstrip())
    elif show == "tail": print("\n".join(out.splitlines()[-tail:]))
    elif callable(show):
        sel = [l for l in out.splitlines() if show(l)]
        print("\n".join(sel) if sel else "(filtered: no matching lines)")
    if p.returncode != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}):\n{out[-2000:]}")
    return out

print("ROOT     =", ROOT)
print("RUN_NAME =", RUN_NAME, "  (16x16 PEs x 4 MAC = 32x32 MAC array)")
print("tech     =", TECH, "| clock =", f"{CLOCK_HZ:g} Hz")
print("outputs ->", OUT)

# fresh start (mirrors the script's cleanup)
for d in (SCSIM, OUT):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
for f in pathlib.Path(HERE, "accelergy_input").glob("*.yaml"):
    f.unlink()
print("cleaned previous artifacts ✔")

In [ ]:
# === What we are modeling ==================================================
import configparser
cp = configparser.ConfigParser(); cp.read(CFG)
ap = cp["architecture_presets"]
print("Systolic array (configs/scale_accel.cfg):")
for k in ["ArrayHeight", "ArrayWidth", "Dataflow",
          "IfmapSramSzkB", "FilterSramSzkB", "OfmapSramSzkB"]:
    print(f"   {k:15s}= {ap.get(k)}")
print("   -> 32x32 = 1024 MAC cells = 16x16 PEs, each a 2x2 (4-MAC) cluster")
print()
print("Workload — one transformer block (M=tokens, N=out features, K=contraction):")
wl = pd.read_csv(TOPO); wl.columns = [c.strip() for c in wl.columns]
wl = wl.dropna(axis=1, how="all")
display(wl)

## Stage 1 — `preprocess.py`: config ➜ a hardware tree Accelergy understands

SCALE-Sim and Accelergy speak different dialects. `preprocess.py` translates the SCALE-Sim
`.cfg` into an **Accelergy architecture** (`architecture.yaml`): a tree of *components*, each with a
**class** (what kind of hardware) and **attributes** (size, width, and crucially `technology: 22nm`).

It also auto-writes the two little driver scripts the next stages call
(`create_action_count.sh`, `run_accelergy.sh`) with the correct run name baked in.

In [ ]:
sh(f'python3 preprocess.py -c "{CFG}" -t "{TOPO}" -p "{SCSIM}" -o "{OUT}" --technology {TECH}')

print("\n--- accelergy_input/architecture.yaml (the hardware tree) ---\n")
arch_txt = open(f"{HERE}/accelergy_input/architecture.yaml").read()
print(arch_txt)

In [ ]:
# What component classes did it instantiate, and how many of each?
arch = yaml.safe_load(arch_txt)
import re
def count_classes(node, acc):
    if isinstance(node, dict):
        if "class" in node and "name" in node:
            m = re.search(r"\[0\.\.(\d+)\]", str(node["name"]))   # PE[0..1023] -> 1024
            acc[node["class"]] = acc.get(node["class"], 0) + (int(m.group(1)) + 1 if m else 1)
        for v in node.values(): count_classes(v, acc)
    elif isinstance(node, list):
        for v in node: count_classes(v, acc)
acc = {}; count_classes(arch, acc)
print("instances by class:")
for k, v in acc.items():
    role = {"custom_DRAM":"off-chip DRAM (CACTI)", "cacti_SRAM":"on-chip SRAM (CACTI, 100% physics)",
            "custom_intmac":"MAC unit (dummy ERT -> your MX netlist)",
            "custom_regfile":"per-PE scratchpad (measured count x netlist)"}.get(k,"")
    print(f"   {v:5d} x {k:18s} {role}")

## Stage 2 — **`scale.py`** (this repo): simulate the array, count the activity

This is **SCALE-Sim** proper (this repo's own `scale.py`). It maps each GEMM onto the 32×32 array and
steps it cycle by cycle, producing reports of **how many cycles** it took and **how many times** each
memory *and each per-PE register file* was read/written. It models *movement*, not energy.

Crucially, scale-sim-v3 instruments the **per-PE scratchpad (register-file) accesses**
(`DETAILED_ACCESS` cols 19–24, via `scalesim/compute/*.get_pe_action_count`) and writes
`REPEAT_CYCLE.csv` — so `pe_regfile` and the SRAM random/repeat split are **measured**, never
fabricated. We pass `-s N` so it skips the multi-GB per-cycle trace dump (reports are still written).

Per layer it reports **Total Cycles** and **Compute Util %** (fraction of the 1024 MAC cells doing
useful work). This step is the slow one (~3–5 min: it walks every cycle to tally per-PE accesses).

In [ ]:
# this repo's scale.py is the activity engine (it emits the per-PE register-file counts).
sh(f'python3 scale.py -c "{CFG}" -t "{TOPO}" -p "{SCSIM}" -i {KIND} -s N',
   cwd=ROOT, show=lambda l: any(k in l for k in
        ["Running Layer", "Total cycles", "Overall utilization", "Run Complete"]))

In [ ]:
# COMPUTE_REPORT — cycles & utilization per layer
comp = pd.read_csv(f"{SC_RAW}/COMPUTE_REPORT.csv"); comp.columns = [c.strip() for c in comp.columns]
comp = comp.dropna(axis=1, how="all")
display(comp[["LayerID", "Total Cycles", "Overall Util %", "Compute Util %"]])

total_cycles = int(comp["Total Cycles"].sum())
print(f"\nTOTAL cycles = {total_cycles:,}  ->  runtime = {total_cycles/CLOCK_HZ*1e3:.3f} ms @ {CLOCK_HZ/1e9:g} GHz")
print(f"avg compute utilization = {comp['Compute Util %'].mean():.1f}%   "
      "(low because the attention GEMMs underfill a 32x32 array)")

## Stage 3 — `create_action_count.py`: reports ➜ per-component action counts

Accelergy needs counts in *its* vocabulary: "component X did action Y, N times." This stage reads
SCALE-Sim's CSV reports and emits `action_count.yaml`, e.g. *"`weights_glb` did 6.3 M reads"*,
*"each `mac` did 1.37 M `mac_random` ops."*

Notice the `data_delta` / `address_delta` tags on memory accesses: Accelergy charges more energy when
bits actually **toggle** (a "random" access, `delta=1`) than when they don't (a repeat, `delta=0`).

In [ ]:
sh("./create_action_count.sh", cwd=HERE, show="tail", tail=4)

ac = yaml.safe_load(open(f"{SC_OUT}/action_count.yaml"))["action_counts"]["local"]
def show_counts(suffix):
    e = next(x for x in ac if x["name"].endswith(suffix))
    print(e["name"])
    for a in e["action_counts"]:
        args = a.get("arguments", {})
        argstr = ", ".join(f"{k}={v}" for k, v in args.items()) if args else ""
        print(f'      {a["name"]:8s} {argstr:40s}  x {a["counts"]:>14,}')
print("\nA few representative components:\n")
for s in ["ifmap_dram", "weights_glb", "PE[0].mac"]:
    show_counts(s); print()

## Stage 4 — `accelergy` + CACTI: turn counts into joules

**Accelergy** now needs an energy for every (component, action). It does **not** know any itself —
instead it holds an *auction*: it asks every installed estimator plug-in *"can you estimate this, and
how confident are you (0–100%)?"* and keeps the highest bid.

- **CACTI** wins every SRAM and DRAM action (accuracy **80%**) → real physics.
- The MAC and scratchpads have no physics model → only the **dummy** plug-in bids (**1%**) → `1.0 pJ` placeholder.

That is why the live log is a wall of *"class X not supported / accuracy 0%"* lines — those are the
**losing bids**, printed because of the `-v 1` verbose flag. We filter to just the winners below.

Outputs: `ERT.yaml` (pJ per action) and `energy_estimation.yaml` (pJ per component = count × ERT).

In [ ]:
sh("./run_accelergy.sh", cwd=HERE,
   show=lambda l: ("accuracy 80%" in l) or ("estimations are saved" in l) or ("reference table is saved" in l))

In [ ]:
# The MAC's ERT — what Accelergy produced on its own (the DUMMY placeholder we will replace)
ert = yaml.safe_load(open(f"{AC_OUT}/ERT.yaml"))["ERT"]["tables"]
mac = next(t for t in ert if t["name"].endswith(".mac"))
print("MAC ERT (Accelergy's own estimate = dummy):")
for a in mac["actions"]:
    print(f'   {a["name"]:12s} -> {a["energy"]} pJ')

# Per-component energy from CACTI (before the netlist swap)
ee = yaml.safe_load(open(f"{AC_OUT}/energy_estimation.yaml"))["energy_estimation"]["components"]
rows = []
for c in ee:
    base = c["name"].split(".")[-1].split("[")[0]
    if base.endswith("dram") or base.endswith("glb"):
        rows.append((c["name"].split(".")[-1], round(c["energy"]/1e9, 4)))
print("\nReal CACTI memory energy per component (mJ):")
display(pd.DataFrame(rows, columns=["component", "energy_mJ"]))

## Stage 5 — `accelergy_eval.py`: inject the MX netlist, get Watts

The final step does three things:
1. Loads `energy/pe_netlist.json` — your synthesized MX array's per-op energy.
2. **Overrides** the dummy `mac` ERT (`1.0 pJ`) with the netlist's `mac_random` (`0.50 pJ`).
3. Re-multiplies counts × energy, buckets into **array / SRAM / DRAM**, and divides by runtime → **average power**.

This is where "SCALE-Sim activity + CACTI memory + your MX array" finally combine into one number.

In [ ]:
print("--- energy/pe_netlist.json (your MX array's per-op energy) ---")
print(open(f"{ROOT}/energy/pe_netlist.json").read())

comp = pd.read_csv(f"{SC_OUT}/COMPUTE_REPORT.csv"); comp.columns = [c.strip() for c in comp.columns]
CYC = int(comp["Total Cycles"].sum())
sh(f'python3 accelergy_eval.py '
   f'--ert "{AC_OUT}/ERT.yaml" --action-count "{SC_OUT}/action_count.yaml" '
   f'--pe-energy "{ROOT}/energy/pe_netlist.json" '
   f'--cycles {CYC} --clock-hz {CLOCK_HZ:.0f} --model {RUN_NAME} --out "{OUT}/energy_{RUN_NAME}"',
   cwd=HERE, show="tail", tail=2)

print("\n=============== FINAL RESULT (ACCELERGY_SUMMARY.md) ===============\n")
print(open(f"{OUT}/energy_{RUN_NAME}/ACCELERGY_SUMMARY.md").read())

## The result, visualized

A dependency-free bar chart of where the energy goes. DRAM dominates — this design is
**memory-bound**, which is the headline takeaway for the power study.

In [ ]:
res = json.load(open(f"{OUT}/energy_{RUN_NAME}/ACCELERGY_ENERGY.json"))
bd  = res["breakdown_pJ"]; tot = sum(bd.values())
src = {"dram": "CACTI (off-chip)", "sram": "CACTI (on-chip)",
       "array_mac": "MX netlist", "pe_regfile": "MX netlist (measured RF)"}
rows = []
for k in ["dram", "sram", "array_mac", "pe_regfile"]:
    v = bd.get(k, 0.0); pct = 100*v/tot
    rows.append([k, src[k], round(v/1e9, 4), round(pct, 1), "█"*int(round(pct/2.5))])
df = pd.DataFrame(rows, columns=["component", "energy source", "mJ", "%", "share"])
display(df)
print(f"\nTOTAL = {tot/1e9:.3f} mJ   |   runtime = {res['runtime_s']*1e3:.3f} ms"
      f"   |   average power = {res['avg_power_W']:.2f} W")

## Bonus: which tool produced each joule?

Every joule in the final number is now one of just two kinds: **CACTI physics** (DRAM + on-chip SRAM)
or **your MX netlist × a simulator-measured access count** (MAC + register file). There is **no
fabricated / dummy term left**: the per-PE RF counts come from scale-sim-v3's measured report, and the
on-chip buffer is a plain CACTI SRAM (storage only) rather than the `smartbuffer_SRAM` compound whose
buffet/counter peripherals used to fall to a flat-1.0 pJ dummy estimate.

In [ ]:
src = {"CACTI physics — DRAM":              bd["dram"],
       "CACTI physics — on-chip SRAM":       bd["sram"],
       "MX netlist x measured count — MAC":  bd["array_mac"],
       "MX netlist x measured count — RF":   bd["pe_regfile"],
       "dummy / fabricated":                 0.0}
tot2 = sum(src.values())
rows = [[k, round(v/1e9, 4), round(100*v/tot2, 1)] for k, v in sorted(src.items(), key=lambda x: -x[1])]
display(pd.DataFrame(rows, columns=["source", "mJ", "%"]))
physics = bd["dram"] + bd["sram"]; netlist = bd["array_mac"] + bd["pe_regfile"]
print(f"\nCACTI physics: {100*physics/tot2:.1f}%   |   netlist x measured count: {100*netlist/tot2:.1f}%"
      f"   |   fabricated/dummy: 0.0%")

## Provenance: measured vs physics vs self-defined

Every number above comes from one of four places — worth knowing which are "measured" and which are
assumptions:

| origin | what it gives | examples |
|---|---|---|
| **SCALE-Sim** (read from reports) | activity *counts* | cycles, DRAM/SRAM accesses, per-PE RF accesses (cols 19-24) |
| **CACTI / Accelergy** (physics) | *energy* per access | SRAM & DRAM pJ/access |
| **Your netlist** (`pe_netlist.json`) | array *energy*/op | MAC 0.5 pJ, spad 0.02 pJ |
| **Self-defined glue** (our code) | a few derivations | ↓ |

The self-defined (analytical) parts that actually move the result:
- **MAC count = Σ cycles** (`create_action_count.py`) — SCALE-Sim reports *cycles*, not MACs; we charge
  one MAC per PE per cycle (un-gated). Sets the *count* side of `array_mac`.
- **All SRAM reads tagged "random"** — this build has no `REPEAT_CYCLE`, so we use CACTI's *toggling*
  (higher) read energy for every read (conservative; total count is still measured).
- **`bank_depth` (KB→rows)** handed to CACTI — the SRAM size CACTI sizes its energy to.
- **Clock = 1 GHz** — sets the **watts** only; energy is clock-independent.

**Idle / leakage is excluded.** `accelergy_eval` zeroes the `idle`/`leak` energy: the idle *counts*
from the report formula `span×dim − accesses` are unreliable (the DRAM idle count is literally
**negative, −22.4 M**), and CACTI returns **0 pJ** for the SRAM idle action here anyway — so it
contributes nothing either way. **Consequence: these are dynamic (switching) energies; standby/leakage
is not modelled.** A full figure would add `leakage_power × runtime`.

## Recap

| stage | tool | in ➜ out | what it really did |
|---|---|---|---|
| 1 | `preprocess.py` | `.cfg` ➜ `architecture.yaml` | described the hardware as a typed component tree @ 22 nm |
| 2 | `scale.py` | topology ➜ `*_REPORT.csv` | simulated the array; counted cycles + memory + per-PE RF accesses |
| 3 | `create_action_count.py` | reports ➜ `action_count.yaml` | re-expressed activity as per-component action counts |
| 4 | `accelergy` + CACTI | ➜ `energy_estimation.yaml` | auctioned each action to an estimator; CACTI won the memories |
| 5 | `accelergy_eval.py` | ➜ `ACCELERGY_SUMMARY.md` | swapped the dummy MAC for your MX netlist → energy & Watts |

**Three ideas to take away**
1. **Separation of concerns** — SCALE-Sim = *how much activity*, CACTI = *energy of a memory access*,
   your netlist = *energy of a MAC*. No single tool does it all.
2. **The auction** — Accelergy's verbose "not supported" spam is just losing bids; the winners are
   CACTI (memories) and your netlist (the array).
3. **Memory dominates** — ~93% of the energy is DRAM traffic, so this workload is memory-bound;
   array/datatype tricks help power only after the DRAM problem is addressed.

Re-run with a different node or clock and watch the numbers move:
```python
TECH = "45nm"      # then re-run stages 1, 4, 5
CLOCK_HZ = 2e8     # then re-run stage 5
```